# 03 — Final Evaluation & Explainability (Fraud Detection)

This notebook is the **final storytelling artifact** for the Insurance Fraud Detection project.

**Goals**
- Quantify final performance on the hold-out test split (with emphasis on **Fraud recall**).
- Compare the tuned XGBoost model against a simple baseline to contextualize gains.
- Explain *why* the model flags fraud using **SHAP** (global + single-case attribution).

> **Threat model note:** insurance fraud is an adversarial, adaptive problem. The evaluation below focuses on reducing *false negatives* (missed fraud) while keeping *false positives* operationally manageable for investigators.

In [ ]:
# --- Setup ---

from __future__ import annotations

import os
import sys
import tempfile
from pathlib import Path

# Avoid matplotlib cache warnings in restricted environments
os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "matplotlib"))

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", context="talk")
np.random.seed(42)

# Make imports work even if you didn't run: `python -m pip install -e .`
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

try:
    import insurance_project  # noqa: F401
except ImportError:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

from insurance_project.paths import find_project_root, data_path, models_path

PROJECT_ROOT = find_project_root(PROJECT_ROOT)
print("Project root:", PROJECT_ROOT)
print("Python:", sys.version.split()[0])

# Optional dependency check (used later for explainability)
try:
    import shap  # noqa: F401
    print("SHAP available")
except ModuleNotFoundError:
    print("SHAP not installed. Run: `pip install -r requirements.txt`")


## 1) Data Loading + Pre-trained Assets

We load the cleaned dataset, reproduce the *same feature engineering/preprocessing contract* used during training, and then load the saved artifacts:
- `best_xgboost_v1.pkl` (tuned model)
- `scaler_v1.pkl` (numeric scaler)

Keeping preprocessing consistent is a core control to prevent silent feature drift between training and production scoring.

In [ ]:
from sklearn.model_selection import train_test_split

from insurance_project.features import add_age_and_tenure_groups
from insurance_project.preprocessing import encode_categorical_features, scale_numerical_features

# Load data (cleaned CSV produced earlier in the pipeline)
df = pd.read_csv(data_path("Automobile_insurance_fraud_cleaned.csv", start=PROJECT_ROOT))
df = add_age_and_tenure_groups(df)

# Load scaler (support both legacy and current filenames)
scaler_path = None
for candidate in ["scaler.pkl", "scaler_v1.pkl"]:
    p = models_path(candidate, start=PROJECT_ROOT)
    if p.exists():
        scaler_path = p
        break
if scaler_path is None:
    raise FileNotFoundError(
        "Could not find scaler in models/. Expected one of: scaler.pkl, scaler_v1.pkl"
    )
scaler = joblib.load(scaler_path)
print("Loaded scaler:", scaler_path.name)

# Apply the same preprocessing contract
df_encoded = encode_categorical_features(df)
df_scaled = scale_numerical_features(df_encoded, is_train=False, scaler=scaler)

# Define feature matrix / target
FEATURE_DROP_COLS = ["fraud_reported", "policy_bind_date", "incident_date"]
X = df_scaled.drop(columns=FEATURE_DROP_COLS, errors="ignore")
y = df_scaled["fraud_reported"].astype(int)

# Recreate the exact split used during modeling for apples-to-apples evaluation
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train shape:", X_train.shape, " | Test shape:", X_test.shape)
print("Test fraud rate:", f"{y_test.mean():.3f}")


In [ ]:
# Load tuned model

model_path = models_path("best_xgboost_v1.pkl", start=PROJECT_ROOT)
if not model_path.exists():
    raise FileNotFoundError(f"Model not found: {model_path}")

best_model = joblib.load(model_path)
print("Loaded model:", model_path.name)
best_model


## 2) Performance Assessment — The Numbers

**Primary KPI: Fraud Recall (Label = 1).**

In insurance economics, a *false negative* is a missed fraud claim that can directly translate into **financial leakage**, potential premium distortion, and follow-on abuse (repeat offenders).

We still monitor precision because every false positive consumes investigator time, but the default posture here is **loss prevention first**: catch more fraud early and use triage to control workload.

In [ ]:
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    precision_recall_curve,
    precision_score,
    recall_score,
)

from insurance_project.modeling import evaluate_model_performance, plot_all_curves
from insurance_project.viz import plot_confusion_matrix

# Core evaluation (uses existing project utilities)
metrics = evaluate_model_performance(best_model, X_test, y_test)
plot_all_curves(best_model, X_test, y_test)

# Extra styling for the confusion matrix (seaborn heatmap)
y_pred = best_model.predict(X_test)
plot_confusion_matrix(y_test, y_pred, title="Confusion Matrix — Tuned XGBoost (Test Set)")

# Tabular classification report (easy to paste into a slide/deck)
report = classification_report(
    y_test, y_pred, target_names=["Normal", "Fraud"], output_dict=True
)
pd.DataFrame(report).T


### Precision–Recall (Fraud-focused)

The PR curve is often more informative than ROC in imbalanced problems because it focuses on **positive-class performance**.

Operationally, the threshold is where economics meets security:
- Higher recall → fewer missed fraud cases (lower leakage)
- Higher precision → fewer false positives (lower investigator load)

Below we compute an illustrative operating point that targets high recall.

In [ ]:
# Precision–Recall curve + an example high-recall operating point

y_prob = best_model.predict_proba(X_test)[:, 1]
precision, recall, thresholds = precision_recall_curve(y_test, y_prob)
pr_auc = average_precision_score(y_test, y_prob)

plt.figure(figsize=(10, 7))
plt.plot(recall, precision, lw=2, label=f"Tuned XGBoost (AP={pr_auc:.3f})")
plt.xlabel("Recall (Fraud)")
plt.ylabel("Precision (Fraud)")
plt.title("Precision–Recall Curve (Test Set)")
plt.grid(alpha=0.25)

# Choose a threshold that achieves at least TARGET_RECALL with the best available precision
TARGET_RECALL = 0.90
if thresholds.size > 0:
    # precision/recall are length thresholds+1; align candidates to thresholds via [:-1]
    candidate_mask = recall[:-1] >= TARGET_RECALL
    if candidate_mask.any():
        candidate_idxs = np.where(candidate_mask)[0]
        best_idx = candidate_idxs[np.argmax(precision[:-1][candidate_idxs])]
        chosen_threshold = float(thresholds[best_idx])

        # Mark on plot
        plt.scatter(
            recall[best_idx],
            precision[best_idx],
            s=120,
            color="black",
            zorder=3,
            label=f"Example threshold={chosen_threshold:.2f}",
        )

        # Compute metrics at this operating point
        y_pred_high_recall = (y_prob >= chosen_threshold).astype(int)
        op_recall = recall_score(y_test, y_pred_high_recall)
        op_precision = precision_score(y_test, y_pred_high_recall)
        print("Example operating point")
        print("- threshold:", f"{chosen_threshold:.3f}")
        print("- fraud recall:", f"{op_recall:.3f}")
        print("- fraud precision:", f"{op_precision:.3f}")

plt.legend(loc="lower left")
plt.show()


## 3) Model Comparison (Context, Not Competition)

A tuned model should be evaluated *relative to a simple baseline* to validate that the added complexity pays off.

We train a lightweight Random Forest baseline on the same training split (with SMOTE applied to mitigate class imbalance) and compare PR curves.

In [ ]:
from insurance_project.modeling import balance_data, plot_pr_comparison, train_baseline_rf

# Baseline training for contextual comparison (kept simple and fast)
X_train_bal, y_train_bal = balance_data(X_train, y_train)
rf_model, _ = train_baseline_rf(X_train_bal, y_train_bal, X_test, y_test)

models_to_compare = {
    "Random Forest (baseline, SMOTE)": rf_model,
    "XGBoost (tuned, saved)": best_model,
}
plot_pr_comparison(models_to_compare, X_test, y_test)


**Interpretation:**

If the tuned model dominates the baseline on the PR curve, it suggests that for the same investigator workload (precision), we can capture more fraud (recall). In production, this enables a **tiered triage** strategy:
- Auto-deny is usually inappropriate; instead, route high-confidence fraud to investigators
- Route borderline cases to additional verification (document requests, call-back, cross-policy checks)
- Continuously monitor drift because fraud tactics evolve

## 4) Model Interpretability — The Why (SHAP)

Fraud models are security controls: they must be **auditable** and defensible. SHAP provides:
- **Global explanation:** which features drive fraud risk overall
- **Local explanation:** why the model flagged a specific claim

We use `TreeExplainer` since the final model is tree-based (XGBoost).

In [ ]:
import shap

# TreeExplainer is optimized for tree-based models
explainer = shap.TreeExplainer(best_model)

# Compute SHAP values for the test set
shap_values = explainer(X_test)

# Global importance (summary plot)
plt.figure(figsize=(11, 8))
shap.summary_plot(shap_values, X_test, show=False, max_display=20)
plt.title("SHAP Summary Plot — Global Feature Impact")
plt.tight_layout()
plt.show()


### Top 5 SHAP Red Flags (Security Lens)

Below are the top drivers by mean absolute SHAP value. Treat them as **risk signals**, not proof of fraud.

From an information security perspective, these features are useful because they can be monitored for:
- abnormal clusters (campaign behavior)
- repeated patterns across identities/policies
- sudden distribution shifts (concept drift / attacker adaptation)

In [ ]:
# Top 5 global risk signals

mean_abs_shap = np.abs(shap_values.values).mean(axis=0)
order = np.argsort(mean_abs_shap)[::-1]

red_flags = pd.DataFrame(
    {
        "feature": X_test.columns[order],
        "mean_abs_shap": mean_abs_shap[order],
    }
).head(5)

red_flags


In [ ]:
# Profiling the top red flags: how do they differ between Fraud vs Normal?

top_features = red_flags["feature"].tolist()

profile_rows = []
for feature in top_features:
    fraud_vals = X_test.loc[y_test == 1, feature]
    normal_vals = X_test.loc[y_test == 0, feature]
    profile_rows.append(
        {
            "feature": feature,
            "mean_abs_shap": float(red_flags.loc[red_flags["feature"] == feature, "mean_abs_shap"].iloc[0]),
            "mean_fraud": float(fraud_vals.mean()),
            "mean_normal": float(normal_vals.mean()),
            "delta(fraud-normal)": float(fraud_vals.mean() - normal_vals.mean()),
        }
    )

profile_df = pd.DataFrame(profile_rows).sort_values("mean_abs_shap", ascending=False)
profile_df

plt.figure(figsize=(10, 5))
sns.barplot(data=profile_df, y="feature", x="delta(fraud-normal)", palette="vlag")
plt.axvline(0, color="black", lw=1)
plt.title("Top 5 red flags — Mean difference (Fraud − Normal)")
plt.xlabel("Mean(Fraud) − Mean(Normal)")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()


**What this means operationally:**

- Features with a large positive delta tend to be *higher* in fraud cases; large negative deltas are *lower* in fraud cases.
- For one-hot features, the delta is an approximate lift in prevalence (fraud vs. normal) and can reveal behavioral clusters.
- Use these red flags to design investigator checklists and to create monitoring alerts for sudden distribution shifts.


## 5) Single-Case Explainability (Waterfall)

We select one **known fraud** example from the test set (where `y_test == 1`) and explain the prediction.

This is the type of artifact that can support:
- investigator notes
- model governance reviews
- dispute handling (why a claim was routed for investigation)

In [ ]:
# Pick one fraud case and explain it locally

fraud_positions = np.where(y_test.values == 1)[0]
if fraud_positions.size == 0:
    raise ValueError("No fraud cases found in the test split. Check stratification/data.")

case_pos = int(fraud_positions[0])
x_case = X_test.iloc[[case_pos]]

case_prob = float(best_model.predict_proba(x_case)[:, 1][0])
case_pred = int(case_prob >= 0.5)
print("Selected case index (positional in X_test):", case_pos)
print("Model output")
print("- P(Fraud=1):", f"{case_prob:.3f}")
print("- Predicted label @0.50:", case_pred)

shap_case = explainer(x_case)
shap.plots.waterfall(shap_case[0], max_display=15)


**How to read the waterfall:**

- Features pushing the score **toward fraud** appear on the positive side; features pushing **toward normal** appear on the negative side.
- This helps investigators focus on *why* the model is suspicious (e.g., claim amount anomalies, timing signals, or inconsistent profile attributes).

In an adversarial setting, explanations are also a double-edged sword: avoid exposing raw decision logic to claimants, and use explanations internally for governance and analyst workflows.

## 6) Security & Economic Insights (The Story)

**Precision vs. Recall is an operational budget decision.**

A practical way to frame it:
- **Cost(false negative)** ≈ expected fraud loss + follow-on risk
- **Cost(false positive)** ≈ investigator time + customer friction

Typical operating model:
1. Choose a high-recall threshold for a *fraud triage queue*.
2. Add secondary controls (rules, document verification, network link analysis) to raise precision downstream.
3. Monitor drift (monthly/quarterly) and retrain if fraud tactics shift.

> Recommended next step: define a target recall based on historical leakage and investigator capacity, then set the threshold accordingly and track it as a KPI.